# 6. Simulasi Hardware & Streaming Citra DOC ke Seeed Grove Vision AI V2
Target Device: **Seeed Grove Vision AI Module V2 (Cortex-M55 + Ethos-U55 NPU)**

Notebook ini menghubungkan laptop ke **Grove Vision AI Module V2** via serial USB-C (`/dev/ttyACM0` @ `921600` baud) untuk menguji model **DOC Sexing on-device**:
1. Mengubah citra DOC menjadi format raw `160x160 RGB uint8` (76.800 byte).
2. Mengirimkannya menggunakan protokol serial berkecepatan tinggi `PP2IR`.
3. Membaca respons inferensi on-device NPU (`PP2_JSON`) dan memvalidasi pengiriman paket I2C 5-Byte ke Arduino Uno R3 / ESP32.


## Langkah 1: Deteksi Serial Port & Pengaturan Koneksi Hardware


In [ ]:
import os
import glob
import time
import json
import struct
import pathlib
import subprocess
import serial
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Menentukan root direktori proyek PP2 secara otomatis dan portabel
cwd = pathlib.Path.cwd().resolve()
if (cwd / "PP2" / "data").exists():
    PP2_DIR = cwd / "PP2"
elif (cwd / "data").exists():
    PP2_DIR = cwd
elif (cwd.parent / "data").exists():
    PP2_DIR = cwd.parent
else:
    PP2_DIR = cwd
    for p in [cwd] + list(cwd.parents):
        if (p / "PP2" / "data").exists():
            PP2_DIR = p / "PP2"
            break
        elif (p / "data").exists():
            PP2_DIR = p
            break

DATA_AUG_DIR = PP2_DIR / "data" / "doc_augmented"
DATA_RAW_DIR = PP2_DIR / "data" / "doc"

def detect_grove_port(default_port="/dev/ttyACM0"):
    """Mendeteksi otomatis port serial /dev/ttyACM* yang aktif."""
    if default_port and os.path.exists(default_port):
        return default_port
    acm_ports = sorted(glob.glob("/dev/ttyACM*"))
    if acm_ports:
        return acm_ports[0]
    return default_port or "/dev/ttyACM0"

def is_picocom_active():
    """Mengecek apakah ada proses picocom yang sedang mengakses serial port."""
    try:
        pids = subprocess.check_output(["pgrep", "-f", "picocom|minicom"], text=True).strip().splitlines()
        return len(pids) > 0
    except subprocess.CalledProcessError:
        return False

SERIAL_PORT = detect_grove_port("/dev/ttyACM0")
BAUDRATE = 921600

print(f"Port Target Grove Vision AI V2 : {SERIAL_PORT}")
print(f"Baudrate Serial               : {BAUDRATE}")
print(f"Port Tersedia di OS           : {os.path.exists(SERIAL_PORT)}")
print(f"Status Picocom Aktif          : {is_picocom_active()} (Tutup picocom Ctrl+A Ctrl+X jika ingin grafik tampil di notebook)")


## Langkah 2: Fungsi Pengiriman Citra DOC (`send_doc_image_to_grove`)


In [ ]:
CONTROLLER_STATE_MAP = {
    (0, "OK"):          ("STATE_SORT_BETINA",     "0° (Kiri - Betina)",  "#2ca02c"),
    (1, "OK"):          ("STATE_SORT_JANTAN",     "180° (Kanan - Jantan)", "#d62728"),
    (2, "BLANK_FRAME"): ("STATE_UNDEFINED_NO_DOC", "90° (Standby/Kosong)", "#7f7f7f"),
    (2, "LOW_CONF"):    ("STATE_UNDEFINED_RAGU",  "90° (Standby/Ragu)",   "#7f7f7f"),
}

def send_doc_image_to_grove(img_input, port=SERIAL_PORT, baud=BAUDRATE, timeout=4.0, show_preview=True):
    """Mengirimkan 1 citra (160x160 RGB uint8) ke Grove Vision AI V2 via USB Serial PP2IR."""
    if isinstance(img_input, (str, pathlib.Path)):
        with Image.open(img_input) as img:
            img_rgb = img.convert("RGB").resize((160, 160), Image.BILINEAR)
            img_name = pathlib.Path(img_input).name
    elif isinstance(img_input, Image.Image):
        img_rgb = img_input.convert("RGB").resize((160, 160), Image.BILINEAR)
        img_name = "Synthetic_DOC_Image"
    else:
        img_rgb = Image.fromarray(np.uint8(img_input)).convert("RGB").resize((160, 160))
        img_name = "Array_DOC_Image"

    rgb_arr = np.asarray(img_rgb, dtype=np.uint8)
    raw_bytes = img_rgb.tobytes()
    assert len(raw_bytes) == 76800, f"Ukuran citra tidak valid: {len(raw_bytes)} (harus 76.800 B)"

    # --- CETAK / TAMPILKAN GAMBAR SEBELUM DIKIRIM KE HARDWARE ---
    if show_preview:
        plt.figure(figsize=(3.2, 3.2))
        plt.imshow(rgb_arr)
        plt.title(f"Akan Dikirim: {img_name}\n(160x160 RGB DOC)", fontsize=9, fontweight="bold")
        plt.axis("off")
        plt.tight_layout()
        plt.show()

    if not os.path.exists(port):
        print(f"[WARN] Port {port} tidak terhubung ke board hardware!")
        return None

    picocom_running = is_picocom_active()
    
    with serial.Serial(port, baud, timeout=timeout) as ser:
        ser.reset_input_buffer()
        ser.reset_output_buffer()

        # Kirim magic header protokol PP2IR + 76.800 bytes
        ser.write(b"PP2IR")
        ser.write(raw_bytes)
        ser.flush()

        t0 = time.time()
        json_line = None

        if not picocom_running:
            while time.time() - t0 < timeout:
                line = ser.readline().decode("utf-8", errors="ignore").strip()
                if "PP2_JSON:" in line:
                    json_str = line.split("PP2_JSON:", 1)[1].strip()
                    try:
                        json_line = json.loads(json_str)
                        break
                    except json.JSONDecodeError:
                        pass
        else:
            print("[INFO] Picocom aktif di terminal: gambar terkirim, hasil tercetak langsung di picocom!")
            return {"status": "sent_picocom", "image_name": img_name}

    if json_line:
        print(f"[RESULT] {img_name} -> Pred: {json_line.get('pred_class')} | Conf: {json_line.get('confidence_pct')}%")
    return json_line


## Langkah 3: Pengujian Kirim Sampel DOC (Female & Male)


In [ ]:
test_female_files = sorted(glob.glob(str(DATA_AUG_DIR / "test" / "Female" / "*.jpg")))
if not test_female_files:
    test_female_files = sorted(glob.glob(str(DATA_RAW_DIR / "Female" / "*.jpg")))

test_male_files = sorted(glob.glob(str(DATA_AUG_DIR / "test" / "Male" / "*.jpg")))
if not test_male_files:
    test_male_files = sorted(glob.glob(str(DATA_RAW_DIR / "Male" / "*.jpg")))

print(f"Kandidat sampel uji: {len(test_female_files)} Female | {len(test_male_files)} Male")

# Uji kirim 1 sampel Female
if test_female_files and os.path.exists(SERIAL_PORT):
    print(f"\n[1] Mengirim sampel Female: {test_female_files[0]}")
    send_doc_image_to_grove(test_female_files[0])

# Uji kirim 1 sampel Male
if test_male_files and os.path.exists(SERIAL_PORT):
    print(f"\n[2] Mengirim sampel Male: {test_male_files[0]}")
    send_doc_image_to_grove(test_male_files[0])
